In [255]:
import gmsh

gmsh.initialize()
gmsh.model.add("mesh1")

In [256]:
# --------------------------------------------------
# 1. GEOMETRY PARAMETERS
# --------------------------------------------------
Lx = 10.0
Ly = 10.0

# 5 layers in z
z0 = 0.0
z1 = 1.5
z2 = 2.0
z3 = 3.0
z4 = 3.5
z5 = 5.0

# Central layer = layer 3 = between z2 and z3
z_middle_central = (z2 + z3) / 2.0   

# Well radius
rw = 0.2

# Opposite-corner well positions

xw1, yw1 = 1.0, 1.0
xw2, yw2 = 9.0, 9.0


In [257]:
# --------------------------------------------------
# 2. CREATE 5 LAYERED BOXES
# --------------------------------------------------
layer1 = gmsh.model.occ.addBox(0, 0, z0, Lx, Ly, z1 - z0)
layer2 = gmsh.model.occ.addBox(0, 0, z1, Lx, Ly, z2 - z1)
layer3 = gmsh.model.occ.addBox(0, 0, z2, Lx, Ly, z3 - z2)
layer4 = gmsh.model.occ.addBox(0, 0, z3, Lx, Ly, z4 - z3)
layer5 = gmsh.model.occ.addBox(0, 0, z4, Lx, Ly, z5 - z4)


In [258]:

# --------------------------------------------------
# 3. CREATE 2 WELLS
#    From top surface down to middle of central layer
# --------------------------------------------------
well_length = z5 - z_middle_central   # from z=5 down to z=2.5

well1 = gmsh.model.occ.addCylinder(xw1, yw1, z5, 0, 0, -well_length, rw)
well2 = gmsh.model.occ.addCylinder(xw2, yw2, z5, 0, 0, -well_length, rw)


In [259]:

# --------------------------------------------------
# 4. FRAGMENT EVERYTHING
#    This makes wells conform with the mesh
# --------------------------------------------------
domain_volumes = [(3, layer1), (3, layer2), (3, layer3), (3, layer4), (3, layer5)]
well_volumes = [(3, well1), (3, well2)]

gmsh.model.occ.fragment(domain_volumes, well_volumes)
gmsh.model.occ.synchronize()


In [260]:
volumes = gmsh.model.getEntities(3)
print("Number of 3D volumes:", len(volumes))

Number of 3D volumes: 11


In [261]:
for s in gmsh.model.getEntities(2):
    com = gmsh.model.occ.getCenterOfMass(2, s[1])
    print("Surface", s[1], "center =", com)

Surface 1 center = (0.0, 5.0, 2.5)
Surface 2 center = (5.0, 0.0, 2.5)
Surface 3 center = (5.0, 5.0, 3.0)
Surface 4 center = (5.0, 10.0, 2.5)
Surface 5 center = (5.0, 5.0, 2.0)
Surface 6 center = (10.0, 5.0, 2.5)
Surface 7 center = (1.0, 1.0, 2.75)
Surface 8 center = (9.0, 9.0, 2.75)
Surface 9 center = (1.0, 1.0, 2.5)
Surface 10 center = (9.0, 9.0, 2.5)
Surface 11 center = (1.0, 1.0, 3.0)
Surface 12 center = (9.0, 9.0, 3.0)
Surface 13 center = (0.0, 5.0, 3.25)
Surface 14 center = (5.0, 0.0, 3.25)
Surface 15 center = (5.0, 5.0, 3.5)
Surface 16 center = (5.0, 10.0, 3.25)
Surface 17 center = (10.0, 5.0, 3.25)
Surface 18 center = (1.0, 1.0, 3.25)
Surface 19 center = (9.0, 9.0, 3.25)
Surface 20 center = (1.0, 1.0, 3.5)
Surface 21 center = (9.0, 9.0, 3.5)
Surface 22 center = (0.0, 5.0, 4.25)
Surface 23 center = (5.0, 0.0, 4.25)
Surface 24 center = (5.0, 5.0, 5.0)
Surface 25 center = (5.0, 10.0, 4.25)
Surface 26 center = (10.0, 5.0, 4.25)
Surface 27 center = (1.0, 1.0, 4.249999999999999)
Surfa

In [262]:

# --------------------------------------------------
# Physical GROUPS

gmsh.model.addPhysicalGroup(3, [9], 101)
gmsh.model.setPhysicalName(3, 101, "Layer_1")

gmsh.model.addPhysicalGroup(3, [6], 102)
gmsh.model.setPhysicalName(3, 102, "Layer_2")

gmsh.model.addPhysicalGroup(3, [3], 103)
gmsh.model.setPhysicalName(3, 103, "Layer_3")

gmsh.model.addPhysicalGroup(3, [2], 104)
gmsh.model.setPhysicalName(3, 104, "Layer_4")

gmsh.model.addPhysicalGroup(3, [1], 105)
gmsh.model.setPhysicalName(3, 105, "Layer_5")

gmsh.model.addPhysicalGroup(3, [10, 7, 4], 201)
gmsh.model.setPhysicalName(3, 201, "Well_1")

gmsh.model.addPhysicalGroup(3, [11, 8, 5], 202)
gmsh.model.setPhysicalName(3, 202, "Well_2")


groups = gmsh.model.getPhysicalGroups()
print(groups)

for dim, tag in gmsh.model.getPhysicalGroups():
    print(dim, tag, gmsh.model.getPhysicalName(dim, tag))

[(3, 101), (3, 102), (3, 103), (3, 104), (3, 105), (3, 201), (3, 202)]
3 101 Layer_1
3 102 Layer_2
3 103 Layer_3
3 104 Layer_4
3 105 Layer_5
3 201 Well_1
3 202 Well_2


In [263]:
well1_vols = gmsh.model.getEntitiesForPhysicalGroup(3, 201)
print(well1_vols)

[ 4  7 10]


In [264]:
for v in [4, 7, 10]:
    print("Volume", v)
    for dim, tag in gmsh.model.getBoundary([(3, v)], oriented=False, recursive=False):
        if dim == 2:
            com = gmsh.model.occ.getCenterOfMass(2, tag)
            print("  Surface", tag, "center =", com)

Volume 4
  Surface 7 center = (1.0, 1.0, 2.75)
  Surface 9 center = (1.0, 1.0, 2.5)
  Surface 11 center = (1.0, 1.0, 3.0)
Volume 7
  Surface 11 center = (1.0, 1.0, 3.0)
  Surface 18 center = (1.0, 1.0, 3.25)
  Surface 20 center = (1.0, 1.0, 3.5)
Volume 10
  Surface 20 center = (1.0, 1.0, 3.5)
  Surface 27 center = (1.0, 1.0, 4.249999999999999)
  Surface 29 center = (1.0, 1.0, 5.0)


In [265]:

# --------------------------------------------------
# Create physical groups for these surfaces:
#The top surface of Well_1 is: Surface 29  ------ center = (1.0, 1.0, 5.0)
# The side surfaces of Well_1 are: 7, 18, 27
# --------------------------------------------------


gmsh.model.addPhysicalGroup(2, [29], 301)
gmsh.model.setPhysicalName(2, 301, "Well1_Top")

gmsh.model.addPhysicalGroup(2, [7], 302)
gmsh.model.setPhysicalName(2, 302, "Well1_Side")

In [266]:
for dim, tag in gmsh.model.getPhysicalGroups(2):
    print(dim, tag, gmsh.model.getPhysicalName(dim, tag))

2 301 Well1_Top
2 302 Well1_Side


In [267]:
#WELL 2

well2_vols = gmsh.model.getEntitiesForPhysicalGroup(3, 202)
print(well2_vols)

[ 5  8 11]


In [268]:
for v in [5, 8, 11]:
    print("Volume", v)
    for dim, tag in gmsh.model.getBoundary([(3, v)], oriented=False, recursive=False):
        if dim == 2:
            com = gmsh.model.occ.getCenterOfMass(2, tag)
            print("  Surface", tag, "center =", com)

Volume 5
  Surface 8 center = (9.0, 9.0, 2.75)
  Surface 10 center = (9.0, 9.0, 2.5)
  Surface 12 center = (9.0, 9.0, 3.0)
Volume 8
  Surface 12 center = (9.0, 9.0, 3.0)
  Surface 19 center = (9.0, 9.0, 3.25)
  Surface 21 center = (9.0, 9.0, 3.5)
Volume 11
  Surface 21 center = (9.0, 9.0, 3.5)
  Surface 28 center = (9.0, 9.0, 4.249999999999999)
  Surface 30 center = (9.0, 9.0, 5.0)


In [269]:
#For Well_2:
#Side surfaces → 8, 19, 28
#Top surface → 30

gmsh.model.addPhysicalGroup(2, [30], 401)
gmsh.model.setPhysicalName(2, 401, "Well2_Top")

gmsh.model.addPhysicalGroup(2, [8], 402)
gmsh.model.setPhysicalName(2, 402, "Well2_Side")

for dim, tag in gmsh.model.getPhysicalGroups(2):
    print(dim, tag, gmsh.model.getPhysicalName(dim, tag))


2 301 Well1_Top
2 302 Well1_Side
2 401 Well2_Top
2 402 Well2_Side


In [270]:

# --------------------------------------------------
# 5. GLOBAL MESH SIZES
#    We first assign a general size to all points
# --------------------------------------------------
all_points = gmsh.model.getEntities(0)
gmsh.model.mesh.setSize(all_points, 10)


In [271]:

# --------------------------------------------------
# 6. REFINEMENT BY LAYER USING BOX FIELDS
#    Small size = more refined
# --------------------------------------------------
# Layer 1 and 5: coarsest
field1 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field1, "VIn", 10)
gmsh.model.mesh.field.setNumber(field1, "VOut", 10)
gmsh.model.mesh.field.setNumber(field1, "XMin", 0)
gmsh.model.mesh.field.setNumber(field1, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field1, "YMin", 0)
gmsh.model.mesh.field.setNumber(field1, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field1, "ZMin", z0)
gmsh.model.mesh.field.setNumber(field1, "ZMax", z1)

field5 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field5, "VIn", 10)
gmsh.model.mesh.field.setNumber(field5, "VOut", 10)
gmsh.model.mesh.field.setNumber(field5, "XMin", 0)
gmsh.model.mesh.field.setNumber(field5, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field5, "YMin", 0)
gmsh.model.mesh.field.setNumber(field5, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field5, "ZMin", z4)
gmsh.model.mesh.field.setNumber(field5, "ZMax", z5)


In [272]:

# Layer 3: most refined
field3 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field3, "VIn", 1)
gmsh.model.mesh.field.setNumber(field3, "VOut", 2)
gmsh.model.mesh.field.setNumber(field3, "XMin", 0)
gmsh.model.mesh.field.setNumber(field3, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field3, "YMin", 0)
gmsh.model.mesh.field.setNumber(field3, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field3, "ZMin", z2)
gmsh.model.mesh.field.setNumber(field3, "ZMax", z3)


In [273]:

# Layer 2 and 4: intermediate
field2 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field2, "VIn", 2)
gmsh.model.mesh.field.setNumber(field2, "VOut", 10)
gmsh.model.mesh.field.setNumber(field2, "XMin", 0)
gmsh.model.mesh.field.setNumber(field2, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field2, "YMin", 0)
gmsh.model.mesh.field.setNumber(field2, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field2, "ZMin", z1)
gmsh.model.mesh.field.setNumber(field2, "ZMax", z2)

field4 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field4, "VIn", 2)
gmsh.model.mesh.field.setNumber(field4, "VOut", 10)
gmsh.model.mesh.field.setNumber(field4, "XMin", 0)
gmsh.model.mesh.field.setNumber(field4, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field4, "YMin", 0)
gmsh.model.mesh.field.setNumber(field4, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field4, "ZMin", z3)
gmsh.model.mesh.field.setNumber(field4, "ZMax", z4)


In [274]:

# --------------------------------------------------
# 7. EXTRA REFINEMENT AROUND WELLS
# --------------------------------------------------
well_field1 = gmsh.model.mesh.field.add("Cylinder")
gmsh.model.mesh.field.setNumber(well_field1, "VIn", 0.2)
gmsh.model.mesh.field.setNumber(well_field1, "VOut", 1)
gmsh.model.mesh.field.setNumber(well_field1, "XCenter", xw1)
gmsh.model.mesh.field.setNumber(well_field1, "YCenter", yw1)
gmsh.model.mesh.field.setNumber(well_field1, "ZCenter", z_middle_central)
gmsh.model.mesh.field.setNumber(well_field1, "XAxis", 0)
gmsh.model.mesh.field.setNumber(well_field1, "YAxis", 0)
gmsh.model.mesh.field.setNumber(well_field1, "ZAxis", well_length)
gmsh.model.mesh.field.setNumber(well_field1, "Radius", 0.8 * 1.5)

well_field2 = gmsh.model.mesh.field.add("Cylinder")
gmsh.model.mesh.field.setNumber(well_field2, "VIn", 0.2)
gmsh.model.mesh.field.setNumber(well_field2, "VOut", 1)
gmsh.model.mesh.field.setNumber(well_field2, "XCenter", xw2)
gmsh.model.mesh.field.setNumber(well_field2, "YCenter", yw2)
gmsh.model.mesh.field.setNumber(well_field2, "ZCenter", z_middle_central)
gmsh.model.mesh.field.setNumber(well_field2, "XAxis", 0)
gmsh.model.mesh.field.setNumber(well_field2, "YAxis", 0)
gmsh.model.mesh.field.setNumber(well_field2, "ZAxis", well_length)
gmsh.model.mesh.field.setNumber(well_field2, "Radius", 0.8 * 1.5)


In [275]:

# --------------------------------------------------
# 8. COMBINE FIELDS
#    Use the minimum size from all refinement fields
# --------------------------------------------------
min_field = gmsh.model.mesh.field.add("Min")
gmsh.model.mesh.field.setNumbers(
    min_field,
    "FieldsList",
    [field1, field2, field3, field4, field5, well_field1, well_field2]
)
gmsh.model.mesh.field.setAsBackgroundMesh(min_field)

# Optional: make Gmsh respect the background field better
gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)


In [276]:
# --------------------------------------------------
# 9. GENERATE 3D MESH
# --------------------------------------------------
gmsh.model.mesh.generate(3)


In [277]:
# --------------------------------------------------
# 10. SAVE
# --------------------------------------------------
gmsh.write("mesh1.msh")


In [278]:
import meshio
import numpy as np

mesh = meshio.read("mesh1.msh")

for i, cell_block in enumerate(mesh.cells):
    if cell_block.type == "tetra":
        vals = mesh.cell_data["gmsh:physical"][i]
        print("Block", i)
        print("Unique physical tags:", np.unique(vals))


Block 4
Unique physical tags: [105]
Block 5
Unique physical tags: [104]
Block 6
Unique physical tags: [103]
Block 7
Unique physical tags: [201]
Block 8
Unique physical tags: [202]
Block 9
Unique physical tags: [102]
Block 10
Unique physical tags: [201]
Block 11
Unique physical tags: [202]
Block 12
Unique physical tags: [101]
Block 13
Unique physical tags: [201]
Block 14
Unique physical tags: [202]


In [279]:
import meshio
import numpy as np

mesh = meshio.read("mesh1.msh")

tetra_cells = []
tetra_phys = []
tetra_geom = []

for i, cell_block in enumerate(mesh.cells):
    if cell_block.type == "tetra":
        tetra_cells.append(cell_block.data)
        tetra_phys.append(mesh.cell_data["gmsh:physical"][i])
        tetra_geom.append(mesh.cell_data["gmsh:geometrical"][i])

all_tetra = np.concatenate(tetra_cells, axis=0)
all_phys = np.concatenate(tetra_phys, axis=0)
all_geom = np.concatenate(tetra_geom, axis=0)

new_mesh = meshio.Mesh(
    points=mesh.points,
    cells=[("tetra", all_tetra)],
    cell_data={
        "gmsh:physical": [all_phys],
        "gmsh:geometrical": [all_geom],
    },
)

meshio.write("mesh1.vtu", new_mesh)

In [280]:
import meshio
import numpy as np

mesh = meshio.read("mesh1.msh")

for i, cell_block in enumerate(mesh.cells):
    vals = mesh.cell_data["gmsh:physical"][i]
    print(f"Block {i}: type = {cell_block.type}, tags = {np.unique(vals)}")


Block 0: type = triangle, tags = [302]
Block 1: type = triangle, tags = [402]
Block 2: type = triangle, tags = [301]
Block 3: type = triangle, tags = [401]
Block 4: type = tetra, tags = [105]
Block 5: type = tetra, tags = [104]
Block 6: type = tetra, tags = [103]
Block 7: type = tetra, tags = [201]
Block 8: type = tetra, tags = [202]
Block 9: type = tetra, tags = [102]
Block 10: type = tetra, tags = [201]
Block 11: type = tetra, tags = [202]
Block 12: type = tetra, tags = [101]
Block 13: type = tetra, tags = [201]
Block 14: type = tetra, tags = [202]


In [281]:
import meshio

mesh = meshio.read("mesh1.msh")

clean_mesh = meshio.Mesh(
    points=mesh.points,
    cells=mesh.cells,
    cell_data=mesh.cell_data
)

meshio.write("mesh1_with_surfaces.vtu", clean_mesh)

In [ ]:
#thresholds

#301 - 402 ----well upper part and botton

In [49]:

# Optional visualization
gmsh.fltk.run()

gmsh.finalize()